# 04 - Transfer learning (EfficientNetB0 / MobileNetV2)

Two-phase training: (1) train a new head on a frozen ImageNet backbone, (2) fine-tune the top backbone layers with a very small learning rate. A GPU (for example Google Colab) is strongly recommended; on CPU each epoch takes minutes.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make the project's `src` package importable from inside notebooks/
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), Path.cwd())
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src import config
from src.utils import find_dataset_root, count_images_per_class

# Point this at data/dummy to try the notebook without the real dataset
DATA_DIR = config.DATA_DIR
ROOT_DATA = find_dataset_root(DATA_DIR)
print("Dataset root:", ROOT_DATA)

from src import train, evaluate
from IPython.display import Image as IPImage, display

In [ ]:
BACKBONE = "efficientnetb0"      # or "mobilenetv2"
WEIGHTS = "imagenet"             # needs internet the first time (downloads ImageNet weights)
EPOCHS = 10                      # phase 1: frozen backbone
FINETUNE_EPOCHS = 10             # phase 2: unfreeze top layers
IMG_SIZE = 224
BATCH_SIZE = 32

## Train

`models/medical_model.keras` is the file the Streamlit app loads, so the best model of this notebook is saved there. Also saved: `models/model_config.json` with the decision threshold chosen on the validation set.

In [ ]:
train.main([
    "--data-dir", str(DATA_DIR),
    "--backbone", BACKBONE,
    "--weights", WEIGHTS,
    "--epochs", str(EPOCHS),
    "--finetune-epochs", str(FINETUNE_EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--img-size", str(IMG_SIZE),
])

In [ ]:
display(IPImage(filename=str(config.OUTPUTS_DIR / "training_history.png")))

The dashed line marks where fine-tuning begins. A small jump or flattening of the validation curves after it is normal.

## Optional: compare backbones

Run the next cell once per backbone, changing `BACKBONE` above, and keep the numbers in a table. Use separate `--model-path` / `--output-dir` values so you do not overwrite `medical_model.keras` with a weaker model.

In [ ]:
# Example: train MobileNetV2 without touching the main model
# train.main([
#     "--data-dir", str(DATA_DIR), "--backbone", "mobilenetv2", "--weights", WEIGHTS,
#     "--epochs", str(EPOCHS), "--finetune-epochs", str(FINETUNE_EPOCHS),
#     "--model-path", str(config.MODELS_DIR / "mobilenetv2.keras"),
#     "--config-path", str(config.MODELS_DIR / "mobilenetv2_config.json"),
#     "--output-dir", str(config.OUTPUTS_DIR / "mobilenetv2"),
# ])